In [ ]:

# !pip install tensorflow==2.12

In [ ]:
# pip install keras-flops

In [ ]:
import os, random, json
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, accuracy_score
from scipy import stats
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import time
print("TensorFlow version:", tf.__version__)

In [ ]:
os.environ['TF_FORCE_GPU_ALLOW_GROWTH'] = 'true'

In [ ]:
# -------------------------------------------------------
# Config
# -------------------------------------------------------
N_SPLITS   = 15     # ⇐ 여러 split 개수 (box plot용)
N_RUNS     = 1           # ⇐ 같은 split에서 반복 학습(원하면 >1)
TRAIN_SEED = 123         # ⇐ 학습 난수 고정 (split 효과만 보려면 고정)
BATCH_SIZE = 32
EPOCHS     = 100
N_CLASSES  = 6
INPUT_KIND = 'xyz'
DATA_TYPE  = 'sliding_doppler'
TRAIN_USER = 'hw'
TEST_USERS = ['hw', 'jh', 'ys']

# === Revision SOTA comparison (이전 결과물과 분리) ===
RUN_TAG = 'revision_sota_20260508'   # 결과물 저장 폴더 식별자


In [ ]:
# -------------------------------------------------------
# Utils: Seed / IO / Normalize / Plot
# -------------------------------------------------------
def set_global_seed(seed: int):
    import os, random
    import numpy as np
    import tensorflow as tf
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    try:
        tf.config.experimental.enable_op_determinism(True)
    except Exception:
        pass

In [ ]:
def load_npy_files(directory):
    return sorted([file for file in os.listdir(directory) if file.endswith('.npy')])

In [ ]:
def choose_channel_and_C(kind):
    if kind == "xtd":        return 0, 1
    elif kind == "ytd":      return 1, 1
    elif kind == "ztd":      return 2, 1
    elif kind == "xtd_ytd":  return slice(0,2), 2
    elif kind == "ytd_ztd":  return slice(1,3), 2
    elif kind == "xtd_ztd":  return [0,2], 2
    elif kind == "xyz":      return slice(0,3), 3
    else: raise ValueError("unknown INPUT_KIND")

In [ ]:
def load_trainval(INPUT_KIND, DATA_TYPE, TRAIN_USER):
    channel, C = choose_channel_and_C(INPUT_KIND)
    directory = f'./dataset_{TRAIN_USER}_timeshift/{DATA_TYPE}/'
    files = load_npy_files(directory)  # 클래스별 .npy 6개라고 가정
    X = np.zeros((6*500, 100, 30, C))
    y = np.zeros((6*500,), dtype=int)
    for i in range(6):
        data = np.load(os.path.join(directory, files[i]))  # shape: (500,100,30,?,3)
        X[500*i:500*(i+1)] = data[:, :, :, :, channel].reshape(500, 100, 30, C)
        y[500*i:500*(i+1)] = i
    return X, y, C

In [ ]:
def normalize_by_train_max(x_train, *others, return_denom=False):
    denom = np.max(np.abs(x_train))
    denom = float(denom) if denom > 0 else 1.0
    outs = [x_train/denom] + [arr/denom for arr in others]
    if return_denom:
        return outs, denom
    return outs if len(outs) > 1 else outs[0]

In [ ]:
def normalize_with_denom(denom, *arrays):
    denom = float(denom) if denom > 0 else 1.0
    return [arr/denom for arr in arrays]

In [ ]:
def save_confusion_matrix(cm, labels, filename, title="Confusion Matrix", normalize=True, as_percent=True):
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1, keepdims=True)
        if as_percent: cm *= 100
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt=".1f" if as_percent else ".2f", cmap='Blues',
                xticklabels=labels, yticklabels=labels, cbar=True)
    plt.xlabel("Predicted"); plt.ylabel("True"); plt.title(title)
    plt.tight_layout(); os.makedirs(os.path.dirname(filename), exist_ok=True)
    plt.savefig(filename, dpi=300); plt.close()

In [ ]:
def evaluate_per_class_accuracy(model, x, y, n_classes=6):
    y_pred = np.argmax(model.predict(x, verbose=0), axis=1)
    acc = []
    for i in range(n_classes):
        idx = np.where(y == i)[0]
        acc.append((y_pred[idx] == y[idx]).mean() if len(idx) else 0.0)
    return np.array(acc)

In [ ]:
# -------------------------------------------------------
# Common Transformer Layers (for CNN-Transformer)
# Reference:
#   - Jin et al. 2024 [22]:  CNN-Transformer & CNN-LSTM
# -------------------------------------------------------
from tensorflow.keras import layers

class AddPositionalEncoding(layers.Layer):
    """Add learnable positional encoding only (no CLS) — used by CNN-Transformer [22]."""
    def __init__(self, num_tokens, embed_dim, name=None):
        super().__init__(name=name)
        self.num_tokens = num_tokens
        self.embed_dim = embed_dim
    def build(self, input_shape):
        self.pos = self.add_weight(
            shape=(1, self.num_tokens, self.embed_dim),
            initializer=tf.keras.initializers.RandomNormal(stddev=0.02),
            trainable=True, name="pos")
    def call(self, x):
        return x + self.pos

class TransformerEncoderBlock(layers.Layer):
    """Pre-norm Transformer encoder: LN -> MHA -> Add ; LN -> MLP(GeLU) -> Add."""
    def __init__(self, d_model, num_heads, mlp_ratio=4.0, drop=0.0, name=None):
        super().__init__(name=name)
        self.ln1 = layers.LayerNormalization(epsilon=1e-6)
        self.attn = layers.MultiHeadAttention(
            num_heads=num_heads, key_dim=d_model // num_heads, dropout=drop)
        self.ln2 = layers.LayerNormalization(epsilon=1e-6)
        self.fc1 = layers.Dense(int(d_model * mlp_ratio), activation="gelu")
        self.drop1 = layers.Dropout(drop)
        self.fc2 = layers.Dense(d_model)
        self.drop2 = layers.Dropout(drop)
    def call(self, x):
        h = self.ln1(x); x = x + self.attn(h, h)
        h = self.ln2(x); h = self.fc2(self.drop1(self.fc1(h)))
        return x + self.drop2(h)


In [ ]:
# -------------------------------------------------------
# CNN-LSTM   (faithful reproduction of Jin et al. 2024 [22])
#
# Paper spec (Sec. IV-A "CNN Block Module"):
#   - "two identical convolutional modules" (parallel branches with independent weights)
#   - each module: 3 conv layers (3x3, channels 64→128→256) + 2 pool layers (2x2)
#   - BN + nonlinear activation after every conv
#   - "Transformer module is replaced with LSTM, rest parts remain unchanged"
#
# Adaptation: input (100, 30, 3) — split 3 channels (X-T-D, Y-T-D, Z-T-D) into
# 3 parallel branches (paper's analog: RTM/DTM => 2 branches for 2 inputs).
# -------------------------------------------------------
def _jin_cnn_branch(x, name):
    """[22] Sec. IV-A: 3 conv (64→128→256, 3x3, BN+ReLU) + 2 pool (2x2)."""
    x = layers.Conv2D(64,  3, padding="same", use_bias=False, name=f"{name}_c1")(x)
    x = layers.BatchNormalization(name=f"{name}_bn1")(x); x = layers.ReLU(name=f"{name}_r1")(x)
    x = layers.MaxPooling2D(2, name=f"{name}_p1")(x)                       # (50, 15, 64)
    x = layers.Conv2D(128, 3, padding="same", use_bias=False, name=f"{name}_c2")(x)
    x = layers.BatchNormalization(name=f"{name}_bn2")(x); x = layers.ReLU(name=f"{name}_r2")(x)
    x = layers.MaxPooling2D(2, name=f"{name}_p2")(x)                       # (25, 7, 128)
    x = layers.Conv2D(256, 3, padding="same", use_bias=False, name=f"{name}_c3")(x)
    x = layers.BatchNormalization(name=f"{name}_bn3")(x); x = layers.ReLU(name=f"{name}_r3")(x)
    return x                                                                # (25, 7, 256)

def build_cnn_lstm(input_shape=(100, 30, 3), num_classes=6,
                   lstm_hidden=(256, 128), dropout=0.2):
    inp = layers.Input(shape=input_shape)
    # Split 3 channels into 3 parallel CNN branches with independent weights
    branches = []
    for i in range(input_shape[-1]):
        bx = layers.Lambda(lambda t, ch=i: t[:, :, :, ch:ch + 1], name=f"split_ch{i}")(inp)
        bx = _jin_cnn_branch(bx, name=f"br{i}")
        branches.append(bx)
    x = layers.Concatenate(axis=-1, name="branch_concat")(branches)        # (25, 7, 768)
    # Flatten spatial → sequence; LSTM replaces Transformer ([22] direct ablation)
    h, w, c = x.shape[1], x.shape[2], x.shape[3]
    x = layers.Reshape((h * w, c), name="flatten_seq")(x)                  # (175, 768)
    x = layers.LSTM(lstm_hidden[0], return_sequences=True,  dropout=dropout,
                    name="lstm1")(x)
    x = layers.LSTM(lstm_hidden[1], return_sequences=False, dropout=dropout,
                    name="lstm2")(x)
    out = layers.Dense(num_classes, activation="softmax", name="cls")(x)
    return tf.keras.Model(inputs=inp, outputs=out, name="CNN_LSTM")


In [ ]:
# -------------------------------------------------------
# CNN-Transformer  (faithful reproduction of Jin et al. 2024 [22])
#
# Paper spec:
#   - CNN_Block: 2 identical conv modules (parallel branches)
#       each: 3 conv (3x3, channels 64→128→256, BN+ReLU) + 2 pool (2x2)
#   - flatten spatial & concat by channel → token sequence
#   - Transformer encoder: 8 stacked, 8 attention heads, dropout=0.2, GeLU MLP
#   - FC_Block: Global Average Pooling + FC  (paper: "params reduced to 1/20")
#
# Adaptation: 3-channel input → 3 parallel branches (paper used 2 for RTM+DTM)
# -------------------------------------------------------
def build_cnn_transformer(input_shape=(100, 30, 3), num_classes=6,
                          embed_dim=512, depth=8, num_heads=8,
                          mlp_ratio=4.0, drop=0.2):
    inp = layers.Input(shape=input_shape)
    # 3 parallel CNN branches (independent weights) — faithful to [22] Sec. IV-A
    branches = []
    for i in range(input_shape[-1]):
        bx = layers.Lambda(lambda t, ch=i: t[:, :, :, ch:ch + 1], name=f"split_ch{i}")(inp)
        bx = _jin_cnn_branch(bx, name=f"br{i}")                              # (25, 7, 256)
        branches.append(bx)
    x = layers.Concatenate(axis=-1, name="branch_concat")(branches)          # (25, 7, 768)
    # Project to embed_dim=512 (paper: 256-per-branch × 2 = 512 sequence dim)
    x = layers.Conv2D(embed_dim, 1, padding="same", use_bias=False,
                      name="proj_to_embed")(x)
    x = layers.BatchNormalization(name="proj_bn")(x)
    x = layers.ReLU(name="proj_relu")(x)
    h, w, d = x.shape[1], x.shape[2], x.shape[3]
    x = layers.Reshape((h * w, d), name="flatten_seq")(x)                    # (175, 512)
    # Positional encoding (paper: "we add the position coding")
    x = AddPositionalEncoding(num_tokens=h * w, embed_dim=embed_dim,
                              name="pos_enc")(x)
    # Stacked Transformer encoders: depth=8, heads=8, dropout=0.2 (paper spec)
    for i in range(depth):
        x = TransformerEncoderBlock(d_model=embed_dim, num_heads=num_heads,
                                    mlp_ratio=mlp_ratio, drop=drop,
                                    name=f"tx{i}")(x)
    x = layers.LayerNormalization(epsilon=1e-6, name="post_ln")(x)
    # FC_Block: Global Average Pooling + FC  (paper Sec. IV-C)
    x = layers.GlobalAveragePooling1D(name="gap")(x)
    out = layers.Dense(num_classes, activation="softmax", name="cls")(x)
    return tf.keras.Model(inputs=inp, outputs=out, name="CNN_Transformer")


In [ ]:
# -------------------------------------------------------
# ShuffleNetV2  (Ma et al., ECCV 2018)
# Reference: efficient lightweight CNN baseline (이전 실험과 비교 위해 추가)
# Adapted to (100, 30, 3) input with scale=1.0
# -------------------------------------------------------
def channel_shuffle(x, groups=2):
    h, w, c = x.shape[1], x.shape[2], x.shape[3]
    assert c % groups == 0, "channels % groups != 0"
    ch = c // groups
    x = tf.reshape(x, [-1, h, w, groups, ch])
    x = tf.transpose(x, [0, 1, 2, 4, 3])
    x = tf.reshape(x, [-1, h, w, c])
    return x

def _conv1x1_bn_relu(x, out_channels):
    x = layers.Conv2D(out_channels, 1, 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)
    return x

def _dwconv3x3_bn(x, stride):
    x = layers.DepthwiseConv2D(3, strides=stride, padding="same", use_bias=False)(x)
    x = layers.BatchNormalization()(x)
    return x

def _shufflenet_v2_unit(x, out_channels):
    assert out_channels % 2 == 0
    c_half = out_channels // 2
    in_c = x.shape[-1]
    if in_c != out_channels:
        x = _conv1x1_bn_relu(x, out_channels)
    x1 = layers.Lambda(lambda t: t[:, :, :, :c_half])(x)
    x2 = layers.Lambda(lambda t: t[:, :, :, c_half:])(x)
    x2 = _conv1x1_bn_relu(x2, c_half)
    x2 = _dwconv3x3_bn(x2, stride=1)
    x2 = _conv1x1_bn_relu(x2, c_half)
    out = layers.Concatenate(axis=-1)([x1, x2])
    out = layers.Lambda(channel_shuffle)(out)
    return out

def _shufflenet_v2_downsample(x, out_channels):
    assert out_channels % 2 == 0
    c_half = out_channels // 2
    b1 = _dwconv3x3_bn(x, stride=2)
    b1 = _conv1x1_bn_relu(b1, c_half)
    b2 = _conv1x1_bn_relu(x, c_half)
    b2 = _dwconv3x3_bn(b2, stride=2)
    b2 = _conv1x1_bn_relu(b2, c_half)
    out = layers.Concatenate(axis=-1)([b1, b2])
    out = layers.Lambda(channel_shuffle)(out)
    return out

def build_shufflenetv2(input_shape=(100, 30, 3), num_classes=6, scale=1.0):
    cfg = {
        0.5: dict(stage2=48,  stage3=96,  stage4=192, conv5=1024),
        1.0: dict(stage2=116, stage3=232, stage4=464, conv5=1024),
        1.5: dict(stage2=176, stage3=352, stage4=704, conv5=1024),
        2.0: dict(stage2=244, stage3=488, stage4=976, conv5=2048),
    }
    ch = cfg[scale]
    inp = layers.Input(shape=input_shape)
    x = layers.Conv2D(24, 3, strides=2, padding="same", use_bias=False)(inp)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.MaxPooling2D(pool_size=3, strides=2, padding="same")(x)
    x = _shufflenet_v2_downsample(x, ch["stage2"])
    for _ in range(3): x = _shufflenet_v2_unit(x, ch["stage2"])
    x = _shufflenet_v2_downsample(x, ch["stage3"])
    for _ in range(7): x = _shufflenet_v2_unit(x, ch["stage3"])
    x = _shufflenet_v2_downsample(x, ch["stage4"])
    for _ in range(3): x = _shufflenet_v2_unit(x, ch["stage4"])
    x = layers.Conv2D(ch["conv5"], 1, use_bias=False)(x)
    x = layers.BatchNormalization()(x); x = layers.ReLU()(x)
    x = layers.GlobalAveragePooling2D()(x)
    out = layers.Dense(num_classes, activation="softmax")(x)
    return tf.keras.Model(inputs=inp, outputs=out, name="ShuffleNetV2")


In [ ]:
# -------------------------------------------------------
# MODEL_ZOO  (Reviewer-requested SOTA comparison + ShuffleNetV2)
#   (A) CNN/LSTM-based         : CNN_LSTM           [Jin et al. 2024 [22]]
#   (B) Transformer-based      : DST                [Jin et al. 2024 [39]] — see srdst.py
#   (C) Hybrid spatial-temporal: CNN_Transformer    [Jin et al. 2024 [22]]
#   (+) Lightweight CNN        : ShuffleNetV2       [Ma et al. ECCV 2018]  ← 이전 실험 결과 재사용
# -------------------------------------------------------
# TRAIN_ZOO: this run에서 새로 학습할 모델만
TRAIN_ZOO = {
    "CNN_LSTM":        build_cnn_lstm,
    "CNN_Transformer": build_cnn_transformer,
}

# MODEL_ZOO: figure/요약 생성 시 쓰는 전체 비교 모델 (ShuffleNetV2는 prior CSV에서 병합)
MODEL_ZOO = {
    "CNN_LSTM":        build_cnn_lstm,
    "CNN_Transformer": build_cnn_transformer,
    "ShuffleNetV2":    build_shufflenetv2,
}


In [ ]:
# -------------------------------------------------------
# Test set loader (per user)
# -------------------------------------------------------
def get_test_data(user, INPUT_KIND, DATA_TYPE, C):
    ch_sel, _ = choose_channel_and_C(INPUT_KIND)
    test_dir = f'./dataset_{user}/{DATA_TYPE}/'
    files = load_npy_files(test_dir)
    files.sort()
    numdata = 50 if user == 'hw' else 30

    X_te = np.zeros((6 * numdata * 10, 100, 30, C))
    y_te = np.zeros((6 * numdata * 10,), dtype=int)

    for i in range(6):
        data = np.load(os.path.join(test_dir, files[i]))
        # data shape assumed: (numdata*10, 100, 30, ?, 3)
        reshaped = data[:, :, :, :, ch_sel].reshape(numdata*10, 100, 30, C)
        X_te[numdata*10*i : numdata*10*(i+1)] = reshaped
        y_te[numdata*10*i : numdata*10*(i+1)] = i
    return X_te, y_te

In [ ]:
# -------------------------------------------------------
# Run experiments across split seeds; collect results
# -------------------------------------------------------
results = []  # for box plot (one row = one (split_seed, model, target) measurement)

# 0) Train/Val 풀 데이터 로드
X_all, y_all, C = load_trainval(INPUT_KIND, DATA_TYPE, TRAIN_USER)
INPUT_SHAPE = (100, 30, C)

for split_seed in range(N_SPLITS):
    # 1) Train/val split
    X_tr, X_va, y_tr, y_va = train_test_split(
        X_all, y_all, test_size=0.1, random_state=split_seed, stratify=y_all
    )

    # 2) 정규화 (train 기준으로 분모 계산 → val/test 동일 적용)
    outs, denom = normalize_by_train_max(X_tr, X_va, return_denom=True)
    X_tr_n, X_va_n = outs

    for model_name, build_fn in TRAIN_ZOO.items():    # 3 SOTA만 학습 (ShuffleNetV2는 prior CSV 병합)
        set_global_seed(TRAIN_SEED)
        model = build_fn(INPUT_SHAPE, N_CLASSES)
        model.compile(optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"])
        model.fit(X_tr_n, y_tr, validation_data=(X_va_n, y_va),
                  epochs=EPOCHS, batch_size=BATCH_SIZE, verbose=0)

        # --- Validation 성능 기록 ---
        val_loss, val_acc = model.evaluate(X_va_n, y_va, verbose=0)
        results.append({
            "target": "val", "user": "val", "split_seed": split_seed,
            "model": model_name, "accuracy": float(val_acc)
        })

        # --- Test 성능 기록 ---
        for user in TEST_USERS:
            X_te, y_te = get_test_data(user, INPUT_KIND, DATA_TYPE, C)
            X_te_n, = normalize_with_denom(denom, X_te)   # <= 여기에서 같은 denom 사용
            te_acc = model.evaluate(X_te_n, y_te, verbose=0)[1]
            results.append({
                "target": "test", "user": user, "split_seed": split_seed,
                "model": model_name, "accuracy": float(te_acc)
            })


# DataFrame으로 정리 + 저장
df = pd.DataFrame(results)
os.makedirs(f'./runs/{INPUT_KIND}/{RUN_TAG}', exist_ok=True)
csv_path = f'./runs/{INPUT_KIND}/{RUN_TAG}/boxplot_results_{INPUT_KIND}.csv'
df.to_csv(csv_path, index=False)
print(f"[Saved] {csv_path}")

In [ ]:
# -------------------------------------------------------
# Merge ShuffleNetV2 results from prior lightweight comparison experiment
# Source: smart_radar_paper/runs/xyz/_summary/boxplot_results_xyz.csv (2025-10, TF 2.12)
# Same protocol: 15 split seeds (0..14), same train/test data, same TRAIN_SEED=123
# -------------------------------------------------------
import os
PRIOR_CSV = '../runs/xyz/_summary/boxplot_results_xyz.csv'
PRIOR_EFF = '../runs/xyz/_summary/model_efficiency_summary.csv'

csv_path = f'./runs/{INPUT_KIND}/{RUN_TAG}/boxplot_results_{INPUT_KIND}.csv'
df_main = pd.read_csv(csv_path)
print(f"[Before merge] {len(df_main)} rows  models={sorted(df_main['model'].unique())}")

if os.path.exists(PRIOR_CSV):
    df_prior = pd.read_csv(PRIOR_CSV)
    df_sn = df_prior[df_prior['model'] == 'ShuffleNetV2'].copy()
    df_main = df_main[df_main['model'] != 'ShuffleNetV2']
    df_main = pd.concat([df_main, df_sn], ignore_index=True)
    df_main.to_csv(csv_path, index=False)
    print(f"[After merge] {len(df_main)} rows  models={sorted(df_main['model'].unique())}")
    print(f"  → ShuffleNetV2: {len(df_sn)} rows merged from prior CSV (TF 2.12, 2025-10)")
else:
    print(f"[WARN] Prior CSV not found at {PRIOR_CSV}")

# Reassign df so downstream cells (boxplots, summary tables) see all 4 models
df = df_main


In [ ]:
# -------------------------------------------------------
# Box plots
# -------------------------------------------------------
import seaborn as sns
import matplotlib.pyplot as plt

# (A) Validation 성능 박스플롯
df_val = df[df["target"] == "val"].copy()

plt.figure(figsize=(9,5))
sns.boxplot(x="model", y="accuracy", data=df_val)
sns.stripplot(x="model", y="accuracy", data=df_val, color=".25", jitter=True, dodge=False)
plt.title("Validation Accuracy across Split Seeds")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f'./runs/{INPUT_KIND}/{RUN_TAG}/boxplot_val_{INPUT_KIND}.png', dpi=300)
plt.close()
print("[Saved] boxplot (val)")

# (B) Test 성능 박스플롯 — 특정 사용자 (예: 'hw')
user_to_plot = 'hw'
df_test_hw = df[(df["target"] == "test") & (df["user"] == user_to_plot)].copy()

plt.figure(figsize=(9,5))
sns.boxplot(x="model", y="accuracy", data=df_test_hw)
sns.stripplot(x="model", y="accuracy", data=df_test_hw, color=".25", jitter=True, dodge=False)
plt.title(f"Test Accuracy across Split Seeds (user={user_to_plot})")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f'./runs/{INPUT_KIND}/{RUN_TAG}/boxplot_test_{user_to_plot}_{INPUT_KIND}.png', dpi=300)
plt.close()
print(f"[Saved] boxplot (test, user={user_to_plot})")

# (C) Test 성능 박스플롯 — 모든 사용자 평균(같은 split에서 평균)
df_test = df[df["target"] == "test"].copy()
# 같은 split_seed × model에서 user 평균을 취해 안정적인 전체 테스트 분포 생성
df_test_avg = (df_test.groupby(["split_seed", "model"], as_index=False)["accuracy"]
                  .mean(numeric_only=True))

plt.figure(figsize=(9,5))
sns.boxplot(x="model", y="accuracy", data=df_test_avg, showfliers=False)
sns.stripplot(x="model", y="accuracy", data=df_test_avg, color=".25", jitter=True)
plt.title("Test Accuracy across Split Seeds (averaged over users)")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f'./runs/{INPUT_KIND}/{RUN_TAG}/boxplot_test_avg_{INPUT_KIND}.png', dpi=300)
plt.close()
print("[Saved] boxplot (test, averaged over users)")

# (D) Test 성능 박스플롯 — 사용자별(hue) 한 장에
df_test = df[df["target"] == "test"].copy()

plt.figure(figsize=(10,6))
ax = sns.boxplot(x="model", y="accuracy", hue="user",
                 data=df_test, showfliers=False, palette="Set2")
# raw 점도 함께 (legend는 boxplot 것만 사용)
sns.stripplot(x="model", y="accuracy", hue="user",
              data=df_test, dodge=True, color=".25", jitter=True, ax=ax, legend=False)

plt.title("Test Accuracy across Split Seeds (per User)")
plt.ylabel("Accuracy")
plt.xlabel("Model")
plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f'./runs/{INPUT_KIND}/{RUN_TAG}/boxplot_test_per_user_{INPUT_KIND}.png', dpi=300)
plt.close()
print("[Saved] boxplot (test, per user)")

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# -------------------------------------------------------
# Helper: IQR 기반 outlier 제거 (컬럼 보존용 transform 방식)
# -------------------------------------------------------
def remove_outliers_iqr(df, group_cols=["model", "user"], value_col="accuracy"):
    if df.empty:
        return df
    grp = df.groupby(group_cols)[value_col]
    q1 = grp.transform(lambda s: s.quantile(0.25))
    q3 = grp.transform(lambda s: s.quantile(0.75))
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    mask = (df[value_col] >= lower) & (df[value_col] <= upper)
    return df[mask]

# -------------------------------------------------------
# 데이터 준비
# -------------------------------------------------------
df_test = df[df["target"] == "test"].copy()


# FIX: keep full model names (some have underscores like CNN_LSTM)
try:
    model_order = [m for m in MODEL_ZOO.keys() if m in df_test["model"].unique()]
except NameError:
    model_order = sorted(df_test["model"].unique())

df_hw = df_test[df_test["user"] == "hw"].copy()
df_hw_no_out = remove_outliers_iqr(df_hw, ["model"], "accuracy")

df_jhys = df_test[df_test["user"].isin(["jh", "ys"])].copy()
df_jhys_no_out = remove_outliers_iqr(df_jhys, ["model", "user"], "accuracy")

df_unknown_avg = (
    df_test[df_test["user"].isin(["jh", "ys"])]
    .groupby(["split_seed", "model"], as_index=False)["accuracy"]
    .mean(numeric_only=True)
)
df_unknown_avg["user"] = "unknown(jh&ys mean)"
df_un = df_unknown_avg.copy()
df_un_no_out = remove_outliers_iqr(df_un, ["model"], "accuracy")

# -------------------------------------------------------
# 시각화 세팅
# -------------------------------------------------------
palette = {"hw": "royalblue","jh": "seagreen","ys": "darkorange","unknown(jh&ys mean)": "gray"}
flier_props = dict(marker='o', markerfacecolor='white',
                   markeredgecolor='black', markersize=4, linestyle='none')

fig, axes = plt.subplots(1, 3, figsize=(18, 6), sharey=False)
axes = axes.ravel()

# (1) hw
sns.boxplot(x="model", y="accuracy", data=df_hw,
            order=model_order, showfliers=True, flierprops=flier_props,
            ax=axes[0], color=palette["hw"])
sns.stripplot(x="model", y="accuracy", data=df_hw_no_out,
              order=model_order, color="black", jitter=True, dodge=False, ax=axes[0], size=3)
axes[0].set_title("Test Accuracy — user A")
axes[0].tick_params(axis='x', rotation=20)
axes[0].set_ylabel("Accuracy")
axes[0].set_xlabel("Model")  # 🔹 x축 라벨 변경

# (2) jh vs ys
sns.boxplot(x="model", y="accuracy", hue="user",
            data=df_jhys, order=model_order, hue_order=["jh", "ys"],
            showfliers=True, flierprops=flier_props,
            ax=axes[1], palette={"jh": palette["jh"], "ys": palette["ys"]})
sns.stripplot(x="model", y="accuracy", hue="user",
              data=df_jhys_no_out, order=model_order, hue_order=["jh", "ys"],
              dodge=True, color="black", jitter=True, ax=axes[1], size=3, legend=False)
axes[1].set_title("Test Accuracy — user B & C")
axes[1].tick_params(axis='x', rotation=20)
axes[1].set_ylabel("Accuracy")
axes[1].set_xlabel("Model")  # 🔹 x축 라벨 변경

handles, labels = axes[1].get_legend_handles_labels()
label_map = {"jh": "user B", "ys": "user C"}
labels = [label_map.get(l, l) for l in labels]
axes[1].legend(handles, labels, title="User")

# (3) unknown(jh&ys mean)
sns.boxplot(x="model", y="accuracy", data=df_un,
            order=model_order, showfliers=True, flierprops=flier_props,
            ax=axes[2], color=palette["unknown(jh&ys mean)"])
sns.stripplot(x="model", y="accuracy", data=df_un_no_out,
              order=model_order, color="black", jitter=True, dodge=False, ax=axes[2], size=3)
axes[2].set_title("Test Accuracy — unknown users (B & C mean)")
axes[2].tick_params(axis='x', rotation=20)
axes[2].set_ylabel("Accuracy")
axes[2].set_xlabel("Model")  # 🔹 x축 라벨 변경

# -------------------------------------------------------
# y축 스케일 조정 (2, 3번 그래프만)
# -------------------------------------------------------
ymin = min(df_jhys["accuracy"].min(), df_un["accuracy"].min())
ymax = 1.0  # 반드시 포함
axes[1].set_ylim(ymin, ymax)
axes[2].set_ylim(ymin, ymax)

plt.tight_layout()
out_path = f'./runs/{INPUT_KIND}/{RUN_TAG}/boxplot_test_hw_jhys_unknown_{INPUT_KIND}.png'
plt.savefig(out_path, dpi=300)
plt.show()
print(f"[Saved] {out_path}")


In [ ]:
# ================================
# 모델별 평균/표준편차 표 (Val)
# ================================
summary_val = (
    df_val.groupby("model")["accuracy"]
    .agg(["mean", "std", "min", "max", "count"])
    .sort_values("mean", ascending=False)
)

# ================================
# Test: user 평균 후 모델별 (즉 split, user 평균 → 모델 평균)
# ================================
summary_test_avg = (
    df_test_avg.groupby("model")["accuracy"]
    .agg(["mean", "std", "min", "max", "count"])
    .sort_values("mean", ascending=False)
)

# ================================
# Test: user별 모델 성능 (개별 user 통계)
# ================================
summary_test_per_user = (
    df_test.groupby(["model", "user"])["accuracy"]
    .agg(["mean", "std", "min", "max", "count"])
    .reset_index()
    .sort_values(["user", "mean"], ascending=[True, False])
)

# ================================
# Print
# ================================
print("\n[Val summary]")
print(summary_val)

print("\n[Test (avg over users) summary]")
print(summary_test_avg)

print("\n[Test per-user summary]")
print(summary_test_per_user)

# ================================
# Save CSVs
# ================================
summary_val.to_csv(f'./runs/{INPUT_KIND}/{RUN_TAG}/summary_val_{INPUT_KIND}.csv')
summary_test_avg.to_csv(f'./runs/{INPUT_KIND}/{RUN_TAG}/summary_test_avg_{INPUT_KIND}.csv')
summary_test_per_user.to_csv(f'./runs/{INPUT_KIND}/{RUN_TAG}/summary_test_per_user_{INPUT_KIND}.csv', index=False)

print(f"[Saved] summary_val_{INPUT_KIND}.csv")
print(f"[Saved] summary_test_avg_{INPUT_KIND}.csv")
print(f"[Saved] summary_test_per_user_{INPUT_KIND}.csv")


In [ ]:
# -----------------------------
# Inference time per dataset
# -----------------------------
def measure_inference_time_on_dataset(model, dataset, n_runs=5):
    """특정 user dataset으로 평균 inference time(ms) 계산"""
    times = []
    for _ in range(n_runs):
        start = time.perf_counter()
        _ = model(dataset, training=False)
        end = time.perf_counter()
        times.append((end - start) * 1000)  # ms
    return np.mean(times)

# -----------------------------
# FLOPs per-sample
# -----------------------------
def _manual_lstm_aware_flops(model):
    """LSTM-friendly FLOPs estimate (TF profiler/keras-flops miss BatchMatMul in LSTM cells)."""
    total = 0
    for layer in model.layers:
        cls = layer.__class__.__name__
        cfg = layer.get_config() if hasattr(layer, "get_config") else {}
        if cls == "Conv2D":
            # 2 * kH * kW * Cin * Cout * Hout * Wout
            try:
                kH, kW = layer.kernel_size; out = layer.output_shape
                Hout, Wout, Cout = out[1], out[2], out[3]
                Cin = layer.input_shape[-1]
                total += 2 * kH * kW * Cin * Cout * Hout * Wout
            except Exception: pass
        elif cls == "DepthwiseConv2D":
            try:
                kH, kW = layer.kernel_size; out = layer.output_shape
                Hout, Wout, C = out[1], out[2], out[3]
                total += 2 * kH * kW * C * Hout * Wout
            except Exception: pass
        elif cls == "Dense":
            try:
                Cin = layer.input_shape[-1]; Cout = layer.units
                # for sequence dense, multiply by sequence length
                shape = layer.output_shape
                T = 1
                for d in shape[1:-1]:
                    if d is not None: T *= d
                total += 2 * Cin * Cout * T
            except Exception: pass
        elif cls == "LSTM":
            try:
                units = layer.units
                in_dim = layer.input_shape[-1]
                T = layer.input_shape[1] or 1
                # 4 gates: (in_dim * units + units * units) MACs * 2 FLOPs/MAC * T
                total += T * 2 * 4 * (in_dim * units + units * units)
            except Exception: pass
        elif cls == "MultiHeadAttention":
            try:
                num_heads = layer.num_heads; key_dim = layer.key_dim
                d_model = num_heads * key_dim
                T = layer.output_shape[1] or 1
                # QKV projections + attention + output projection
                total += T * (3 * 2 * d_model * d_model + 2 * 2 * T * d_model + 2 * d_model * d_model)
            except Exception: pass
    return int(total)

def try_get_flops_per_sample(model, input_shape):
    try:
        from keras_flops import get_flops
        return int(get_flops(model, batch_size=1))
    except Exception as e:
        print("[keras-flops] 실패:", repr(e))

    # fallback: TF profiler
    try:
        from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2
        @tf.function
        def forward(x): return model(x, training=False)
        concrete = forward.get_concrete_function(tf.TensorSpec([1, *input_shape], tf.float32))
        frozen_func = convert_variables_to_constants_v2(concrete)
        graph_def = frozen_func.graph.as_graph_def()

        tf.compat.v1.reset_default_graph()
        with tf.Graph().as_default():
            tf.graph_util.import_graph_def(graph_def, name='')
            run_meta = tf.compat.v1.RunMetadata()
            opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()
            flops = tf.compat.v1.profiler.profile(graph=None, run_meta=run_meta,
                                                  cmd='op', options=opts)
        return int(flops.total_float_ops) if flops else None
    except Exception as e:
        print("[tf.profiler] 실패:", repr(e))

    # FIX: Manual LSTM-aware fallback (handles BatchMatMul in LSTM cells)
    try:
        f = _manual_lstm_aware_flops(model)
        if f > 0:
            return f
    except Exception as e:
        print("[manual fallback] 실패:", repr(e))
    return None

# -----------------------------
# Collect stats per model
# -----------------------------
def get_model_stats(build_fn, input_shape, num_classes, user_inputs):
    model  = build_fn(input_shape, num_classes)
    params = int(model.count_params())
    flops  = try_get_flops_per_sample(model, input_shape)   # per-sample ops

    # 각 user dataset으로 inference time 측정
    inf_times = []
    for user, data in user_inputs.items():
        inf_t = measure_inference_time_on_dataset(model, data, n_runs=10)
        inf_times.append(inf_t)
    inf_ms = np.mean(inf_times)  # 모든 user 평균(ms)

    return params, flops, inf_ms

# -----------------------------
# Collect stats per model  (✅ 한 모델 = 한 행, 3개 지표를 컬럼에)
# -----------------------------
stats_rows = []
for model_name, build_fn in MODEL_ZOO.items():
    user_inputs = {
        "hw": tf.random.normal([1, *INPUT_SHAPE], dtype=tf.float32),
        "jh": tf.random.normal([1, *INPUT_SHAPE], dtype=tf.float32),
        "ys": tf.random.normal([1, *INPUT_SHAPE], dtype=tf.float32),
    }
    params, flops, inf_ms = get_model_stats(build_fn, INPUT_SHAPE, N_CLASSES, user_inputs)

    stats_rows.append({
        "model": model_name,
        "Params(M)": params / 1e6,
        "FLOPs(M)": (flops / 1e6) if flops is not None else np.nan,  # flops 실패 시 NaN
        "Inference Time(ms)": float(inf_ms),
    })

# MODEL_ZOO 순서 유지(선택)
model_order = list(MODEL_ZOO.keys())
df_stats = (pd.DataFrame(stats_rows)
              .set_index("model")
              .reindex(model_order))

# -----------------------------
# 출력 + 저장 (표가 짧고 읽기 쉬움)
# -----------------------------
print("\n[Model Efficiency Stats]")
print(df_stats.round(4))
outdir = f'./runs/{INPUT_KIND}/{RUN_TAG}'
os.makedirs(outdir, exist_ok=True)
df_stats.to_csv(f"{outdir}/model_efficiency_summary.csv")
print(f"[Saved] {outdir}/model_efficiency_summary.csv")

# -----------------------------
# 시각화
# -----------------------------
metrics = ["Params(M)", "FLOPs(M)", "Inference Time(ms)"]

plt.figure(figsize=(12, 5))
for idx, metric in enumerate(metrics, 1):
    plt.subplot(1, 3, idx)
    # seaborn이 NaN은 자동으로 무시하고 그립니다
    sns.barplot(x=df_stats.index, y=df_stats[metric].values)
    plt.title(metric)
    plt.xticks(rotation=20)
plt.tight_layout()
plt.savefig(f"{outdir}/barplot_model_stats.png", dpi=300)
plt.show()
print(f"[Saved] {outdir}/barplot_model_stats.png")


In [ ]:
# # ===========================
# # Model stats (Params / MFLOPs / Inference time)
# # ===========================
# import os, io, time, contextlib, logging
# import numpy as np
# import pandas as pd
# import matplotlib.pyplot as plt
# import seaborn as sns
# import tensorflow as tf

# # 콘솔 로그 최소화
# os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # 0:all,1:warn,2:error,3:fatal
# tf.get_logger().setLevel(logging.ERROR)

# # -----------------------------
# # Inference time (pure forward)
# # -----------------------------
# def measure_inference_time(model, input_shape, n_runs=30, batch_size=1):
#     """순수 forward 경로만 측정 (per-run sec)"""
#     dummy = tf.random.normal([batch_size, *input_shape], dtype=tf.float32)
#     # Warm-up (그래프/JIT 준비)
#     _ = model(dummy, training=False)
#     _ = model(dummy, training=False)
#     start = time.perf_counter()
#     for _ in range(n_runs):
#         _ = model(dummy, training=False)
#     end = time.perf_counter()
#     return (end - start) / n_runs  # seconds

# # -----------------------------
# # FLOPs (per-sample, with fallback TF profiler)
# # -----------------------------
# def try_get_flops_per_sample(model, input_shape):
#     """
#     1샘플(per-sample) 기준 FLOPs(ops) 반환.
#     1) keras-flops 우선 시도 (없으면 스킵)
#     2) 실패 시 TF v1 profiler 사용
#     """
#     # A) keras-flops
#     try:
#         from keras_flops import get_flops
#         return int(get_flops(model, batch_size=1))  # per-sample
#     except Exception as e:
#         print("[keras-flops] 실패:", repr(e))

#     # B) TF profiler fallback
#     try:
#         from tensorflow.python.framework.convert_to_constants import convert_variables_to_constants_v2

#         @tf.function
#         def forward(x):
#             return model(x, training=False)

#         concrete = forward.get_concrete_function(tf.TensorSpec([1, *input_shape], tf.float32))
#         frozen_func = convert_variables_to_constants_v2(concrete)
#         graph_def = frozen_func.graph.as_graph_def()

#         tf.compat.v1.reset_default_graph()
#         with tf.Graph().as_default():
#             tf.graph_util.import_graph_def(graph_def, name='')
#             run_meta = tf.compat.v1.RunMetadata()
#             opts = tf.compat.v1.profiler.ProfileOptionBuilder.float_operation()

#             # stdout 캡처해 터미널 리포트 억제
#             buf = io.StringIO()
#             with contextlib.redirect_stdout(buf):
#                 flops = tf.compat.v1.profiler.profile(
#                     graph=None, run_meta=run_meta, cmd='op', options=opts
#                 )
#         return int(flops.total_float_ops) if flops is not None else None
#     except Exception as e:
#         print("[tf.profiler] 실패:", repr(e))
#         return None

# # -----------------------------
# # Collect stats per model
# # -----------------------------
# def get_model_stats(build_fn, input_shape, num_classes):
#     model  = build_fn(input_shape, num_classes)
#     params = int(model.count_params())
#     flops  = try_get_flops_per_sample(model, input_shape)   # per-sample FLOPs (ops)
#     inf_s  = measure_inference_time(model, input_shape, n_runs=30, batch_size=1)
#     return params, flops, inf_s

# stats_results = []
# for model_name, build_fn in MODEL_ZOO.items():
#     params, flops, inf_s = get_model_stats(build_fn, INPUT_SHAPE, N_CLASSES)

#     # 원본 값 저장 (CSV용)
#     stats_results.append({"model": model_name, "metric": "Params",             "value": params})
#     if flops is not None:
#         stats_results.append({"model": model_name, "metric": "FLOPs",          "value": flops})   # per-sample ops
#     else:
#         print(f"[WARN] FLOPs 계산 실패: {model_name} → N/A")
#     stats_results.append({"model": model_name, "metric": "Inference Time (s)", "value": inf_s})

# df_stats_raw = pd.DataFrame(stats_results)

# # -----------------------------
# # Unit scaling (Params=M, FLOPs=MFLOPs, Time=ms)
# # -----------------------------
# df_stats = df_stats_raw.copy()

# def scale(row):
#     m, v = row["metric"], row["value"]
#     if m == "Params":
#         return v / 1e6           # M params
#     elif m == "FLOPs":
#         return v / 1e6           # MFLOPs  (연산량)
#     elif m == "Inference Time (s)":
#         return v * 1e3           # ms
#     return v

# df_stats["value_scaled"] = df_stats.apply(scale, axis=1)
# df_stats["unit"] = df_stats["metric"].map({
#     "Params": "Millions",
#     "FLOPs": "MFLOPs",
#     "Inference Time (s)": "ms",
# }).fillna("")

# # -----------------------------
# # Plot (only panels that have data)
# # -----------------------------
# desired_order = ["Params", "FLOPs", "Inference Time (s)"]
# metrics_present = [m for m in desired_order if m in df_stats["metric"].unique()]
# if not metrics_present:
#     raise RuntimeError("No metrics to plot. Check df_stats.")

# fig, axes = plt.subplots(1, len(metrics_present), figsize=(6*len(metrics_present), 6))
# if len(metrics_present) == 1:
#     axes = [axes]

# ylabels = {
#     "Params": "Params (Millions)",
#     "FLOPs": "FLOPs (MFLOPs)",
#     "Inference Time (s)": "Inference Time (ms)",
# }

# for ax, m in zip(axes, metrics_present):
#     df_m = df_stats[df_stats["metric"] == m]
#     # 박스 + 점
#     sns.boxplot(x="model", y="value_scaled", data=df_m, ax=ax, showfliers=False)
#     sns.stripplot(x="model", y="value_scaled", data=df_m, color=".25", jitter=False, ax=ax)
#     ax.set_title(m)
#     ax.set_ylabel(ylabels.get(m, "Value"))
#     ax.set_xlabel("model")
#     ax.tick_params(axis="x", labelrotation=20)

# plt.tight_layout()
# outdir = f'./runs/{INPUT_KIND}/{RUN_TAG}'
# os.makedirs(outdir, exist_ok=True)
# plt.savefig(f'{outdir}/boxplot_model_stats.png', dpi=300)
# plt.show()

# # -----------------------------
# # CSV 저장 + 요약 피벗 프린트
# # -----------------------------
# df_stats_raw.to_csv(f"{outdir}/model_stats_raw.csv", index=False)
# df_stats.to_csv(f"{outdir}/model_stats_scaled.csv", index=False)

# pivot = df_stats.pivot_table(index="model", columns="metric", values="value_scaled", aggfunc="first")
# pivot = pivot.rename(columns={
#     "Params": "Params (M)",
#     "FLOPs": "FLOPs (MFLOPs)",
#     "Inference Time (s)": "Time (ms)"
# })
# print("\n[Summary Table — scaled units]")
# print(pivot.round(3))

# print(f"\n[Saved] {outdir}/boxplot_model_stats.png")
# print(f"[Saved] {outdir}/model_stats_raw.csv")
# print(f"[Saved] {outdir}/model_stats_scaled.csv")


In [ ]:
# =========================================
# MFLOPs vs. Accuracy (untrained users jh, ys)
# =========================================
from matplotlib.lines import Line2D

outdir = f'./runs/{INPUT_KIND}/{RUN_TAG}'
os.makedirs(outdir, exist_ok=True)

# 1) FLOPs/Params: 방금 만든 df_stats(와이드: index=model)에서 바로 추출
mf = df_stats["FLOPs(M)"].rename("MFLOPs").reset_index()      # [model, MFLOPs]
pm = df_stats["Params(M)"].rename("Params_M").reset_index()   # [model, Params_M]

# 2) Accuracy: 원본 df로부터 untrained users(jh, ys)만 사용
df_test_full = df[df["target"] == "test"].copy()                    # 풀네임 유지!
df_test_bc   = df_test_full[df_test_full["user"].isin(["jh","ys"])]

df_test_avg_bc = (df_test_bc
                  .groupby(["split_seed","model"], as_index=False)["accuracy"]
                  .mean(numeric_only=True))
acc_per_model  = (df_test_avg_bc
                  .groupby("model", as_index=False)["accuracy"]
                  .mean(numeric_only=True)
                  .rename(columns={"accuracy":"Accuracy"}))

# 3) 병합(outer로 진단 가능) → 숫자 필요한 행만 남김
df_comp = (mf.merge(pm, on="model", how="outer")
             .merge(acc_per_model, on="model", how="outer"))

# 숫자형 보장 & 결측 제거(숫자 없는 모델은 산점도 좌표를 찍을 수 없음)
for c in ["MFLOPs","Params_M","Accuracy"]:
    df_comp[c] = pd.to_numeric(df_comp[c], errors="coerce")
df_comp = df_comp.dropna(subset=["MFLOPs","Params_M","Accuracy"]).reset_index(drop=True)

# 4) 산점도
plt.figure(figsize=(8,6))

# 팔레트: 모델별 고정 색상 (사용자 지정)
MODEL_COLORS = {
    "CNN_LSTM":        "#1f3b73",   # navy
    "CNN_Transformer": "#808080",   # gray
    "ShuffleNetV2":    "#d62728",   # red (ShuffleNetV2 distinct)
}
model_names = df_comp["model"].tolist()
palette = {m: MODEL_COLORS.get(m, "#444444") for m in model_names}

# 버블 크기(면적)
size_min, size_max = 60, 420
ax = sns.scatterplot(
    data=df_comp,
    x="MFLOPs", y="Accuracy",
    hue="model",
    size="Params_M", sizes=(size_min, size_max),
    palette=palette,
    legend=False
)

# 1차 추세선
if len(df_comp) >= 2:
    x = df_comp["MFLOPs"].values
    y = df_comp["Accuracy"].values
    p = np.polyfit(x, y, 1)
    xx = np.linspace(x.min(), x.max(), 100)
    yy = p[0]*xx + p[1]
    plt.plot(xx, yy, "--", color="gray", alpha=0.6)

plt.title("MFLOPs vs. Test Accuracy (Untrained Users jh, ys)")
plt.xlabel("FLOPs (MFLOPs, per-sample)")
plt.ylabel("Accuracy")
# 필요하면 고정: plt.ylim(0.0, 1.0)
plt.tight_layout()

# 5) 커스텀 레전드(모델명 + Params(M), 버블 크기/색 동일)
vmin, vmax = df_comp["Params_M"].min(), df_comp["Params_M"].max()
def map_size(v):
    if vmax == vmin:
        return 0.5*(size_min + size_max)
    return size_min + (v - vmin) * (size_max - size_min) / (vmax - vmin)

df_leg = df_comp.sort_values("Params_M", ascending=False)
handles = []
for _, row in df_leg.iterrows():
    m  = row["model"]
    pM = float(row["Params_M"])
    ms = np.sqrt(map_size(pM))  # Line2D는 지름(points) 기준
    h = Line2D([], [], marker='o', linestyle='',
               markersize=ms,
               markerfacecolor=palette[m],
               markeredgecolor='white', markeredgewidth=0.6,
               label=f"{m} ({pM:.2f}M)")
    handles.append(h)

ax.legend(handles=handles, title="Model (Params M)",
          bbox_to_anchor=(0.61, 0.45), loc="upper left",
          frameon=True, handleheight=1.5, labelspacing=1.0, borderaxespad=0.5)

# 저장 & 표 확인
out_path = os.path.join(outdir, f"scatter_mflops_vs_accuracy_untrained_{INPUT_KIND}.png")
plt.savefig(out_path, dpi=300)
plt.show()
print(f"[Saved] {out_path}")

print("\n[Table] MFLOPs / Params(M) / Accuracy (users jh, ys only)")
print(df_comp.sort_values("Accuracy", ascending=False).reset_index(drop=True).round(4))


In [ ]:
# ============================
# 1×3 Combined Figure with grids
#   (0) jh vs ys
#   (1) unknown (jh&ys mean)
#   (2) MFLOPs vs Accuracy (bubble = Params(M)) + vertical FLOPs threshold
# ============================

import os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D

# ---------- 팔레트 ----------
palette_users = {"jh": "seagreen", "ys": "darkorange"}  # 사용자용
unknown_color = "gray"                                  # unknown용

# ---------- 유틸: df_stats에서 MFLOPs/Params_M 안전 추출 ----------
# df_stats는 환경에 따라 아래 중 하나일 수 있음:
#  A) 인덱스 = ["Params(M)", "FLOPs(M)", "Inference Time(ms)"], 컬럼 = 모델명  (권장 형태)
#  B) 컬럼에 "Params(M)"/"FLOPs(M)"가 있고 인덱스가 모델명인 형태
#  C) 롱포맷(예: ["model","metric","value"]) 혹은 저장된 CSV만 있는 경우
def _extract_metric_table(df_stats, metric_hint_lower, out_name, outdir):
    """
    df_stats에서 metric_hint_lower(예: 'flop', 'param')을 포함하는
    인덱스/컬럼/롱포맷에서 해당 값을 [model, <out_name>] 형태로 반환.
    """
    # Case A: 인덱스에서 찾기 (예: 'FLOPs(M)'가 인덱스 라벨)
    try:
        idx_candidates = [
            idx for idx in df_stats.index
            if isinstance(idx, str) and (metric_hint_lower in idx.lower())
        ]
        if idx_candidates:
            s = df_stats.loc[idx_candidates[0]]
            return (s.rename(out_name)
                     .reset_index()
                     .rename(columns={"index": "model"}))
    except Exception:
        pass

    # Case B: 컬럼에서 찾기 (예: 'FLOPs(M)'가 컬럼 라벨)
    try:
        col_candidates = [
            c for c in df_stats.columns
            if isinstance(c, str) and (metric_hint_lower in c.lower())
        ]
        if col_candidates:
            col = col_candidates[0]
            if "model" in df_stats.columns:   # 모델명이 컬럼에 있을 때
                return df_stats[["model", col]].rename(columns={col: out_name})
            else:                              # 모델명이 인덱스일 때
                return (df_stats[col]
                        .rename(out_name)
                        .reset_index()
                        .rename(columns={"index": "model"}))
    except Exception:
        pass

    # Case C: 저장된 CSV에서 불러와서 시도
    try_paths = [
        os.path.join(outdir, "model_efficiency_summary.csv"),
        os.path.join(outdir, "model_stats_scaled.csv"),
        os.path.join(outdir, "model_stats_raw.csv"),
    ]
    for pth in try_paths:
        if os.path.exists(pth):
            try:
                df_csv = pd.read_csv(pth)
                # 롱포맷: ["model","metric","value"] 또는 ["model","metric","value_scaled"]
                if {"model", "metric", "value"}.issubset(df_csv.columns):
                    msk = df_csv["metric"].astype(str).str.lower().str.contains(metric_hint_lower)
                    tmp = df_csv.loc[msk, ["model", "value"]].rename(columns={"value": out_name})
                    if not tmp.empty:
                        return tmp
                if {"model", "metric", "value_scaled"}.issubset(df_csv.columns):
                    msk = df_csv["metric"].astype(str).str.lower().str.contains(metric_hint_lower)
                    tmp = df_csv.loc[msk, ["model", "value_scaled"]].rename(columns={"value_scaled": out_name})
                    if not tmp.empty:
                        return tmp
                # 와이드포맷: "FLOPs(M)" 같은 컬럼이 있을 수도 있음
                col_candidates = [
                    c for c in df_csv.columns
                    if isinstance(c, str) and (metric_hint_lower in c.lower())
                ]
                if col_candidates:
                    col = col_candidates[0]
                    if "model" in df_csv.columns:
                        return df_csv[["model", col]].rename(columns={col: out_name})
                    else:
                        # 모델이 인덱스일 수 있으므로 인덱스 복구
                        if df_csv.index.name == "model":
                            return (df_csv[col]
                                    .rename(out_name)
                                    .reset_index()
                                    .rename(columns={"index": "model"}))
            except Exception:
                pass

    raise KeyError(f"df_stats에서 '{metric_hint_lower}'(을)를 찾을 수 없습니다. "
                   f"df_stats 구조 또는 저장된 CSV를 확인하세요.")

# ---------- FLOPs/Params/Accuracy 테이블 ----------
outdir = f'./runs/{INPUT_KIND}/{RUN_TAG}'

# df_stats로부터 안전하게 MFLOPs/Params_M 추출
mf = _extract_metric_table(df_stats, metric_hint_lower="flop",  out_name="MFLOPs",  outdir=outdir)
pm = _extract_metric_table(df_stats, metric_hint_lower="param", out_name="Params_M", outdir=outdir)

# Accuracy: 원본 df에서 untrained users (jh, ys)
df_test_full = df[df["target"] == "test"].copy()
df_test_bc   = df_test_full[df_test_full["user"].isin(["jh","ys"])]
df_test_avg_bc = (df_test_bc.groupby(["split_seed","model"], as_index=False)["accuracy"]
                           .mean(numeric_only=True))
acc_per_model  = (df_test_avg_bc.groupby("model", as_index=False)["accuracy"]
                               .mean(numeric_only=True)
                               .rename(columns={"accuracy":"Accuracy"}))

# 병합 + 숫자형 보장 + 결측 제거
df_comp = (mf.merge(pm, on="model", how="outer")
             .merge(acc_per_model, on="model", how="outer"))
for c in ["MFLOPs","Params_M","Accuracy"]:
    df_comp[c] = pd.to_numeric(df_comp[c], errors="coerce")
df_comp = df_comp.dropna(subset=["MFLOPs","Params_M","Accuracy"]).reset_index(drop=True)

# ---------- 1×3 캔버스 (y축 공유로 스케일 통일) ----------
fig, axes = plt.subplots(1, 3, figsize=(22, 7),)  # 🔹 모든 서브플롯 y축 공유

# (0) jh vs ys
sns.boxplot(x="model", y="accuracy", hue="user",
            data=df_jhys, order=model_order, hue_order=["jh","ys"],
            showfliers=True, flierprops=flier_props,
            ax=axes[0], palette=palette_users)
sns.stripplot(x="model", y="accuracy", hue="user",
              data=df_jhys_no_out, order=model_order, hue_order=["jh","ys"],
              dodge=True, color="black", jitter=True, ax=axes[0], size=3, legend=False)
axes[0].set_title("Test Accuracy of Untrained User B and C", fontsize=17)
axes[0].tick_params(axis='x', rotation=20, labelsize=15)
axes[0].tick_params(axis='y', labelsize=15)
axes[0].set_ylabel("Accuracy", fontsize=17)
axes[0].set_xlabel("Model",fontsize=17)
# legend 이름 교체
handles, labels = axes[0].get_legend_handles_labels()
label_map = {"jh": "User B", "ys": "User C"}
labels = [label_map.get(l, l) for l in labels]

axes[0].legend(
    handles, labels,
    loc="lower right",
    title="Untrained User",          # 타이틀 쓰는 경우
    fontsize=14,           # 라벨 폰트 크기
    title_fontsize=14,    # 타이틀 폰트 크기
    handlelength=1.6,      # (옵션) 핸들 길이
    handletextpad=0.6,     # (옵션) 핸들-텍스트 간격
    borderaxespad=0.4      # (옵션) 경계 여백
)


# (1) unknown(jh&ys mean)
sns.boxplot(x="model", y="accuracy", data=df_un,
            order=model_order, showfliers=True, flierprops=flier_props,
            ax=axes[1], color=unknown_color)
sns.stripplot(x="model", y="accuracy", data=df_un_no_out,
              order=model_order, color="black", jitter=True, dodge=False, ax=axes[1], size=3)
axes[1].set_title("Average Test Accuracy of Untrained User B and C", fontsize=17)
axes[1].tick_params(axis='x', rotation=20, labelsize=15)
axes[1].tick_params(axis='y', labelsize=15)
axes[1].set_xlabel("Model", fontsize = 17)       
axes[1].set_ylabel("Accuracy", fontsize = 17)# 🔹 y라벨 강제
axes[1].tick_params(axis='y', labelleft=True)  # 🔹 y눈금 강제

# (2) MFLOPs vs Accuracy (bubble = Params(M)) + FLOPs 임계선(수직선)
ax = axes[2]

# 🔧 세 번째 축에 y라벨/눈금/제목을 강제로 표시
ax.set_ylabel("Accuracy", fontsize = 17)
ax.tick_params(axis='x', labelsize=15)  # x축 눈금 크기
ax.tick_params(axis='y', labelsize=15)
ax.set_xlabel("MFLOPs", fontsize=17)
ax.set_title("MFLOPs vs. Average Test Accuracy", fontsize=17)

# FIX: data-driven xlim with log scale (MFLOPs span 1+ decade)
ax.set_xscale("log")
xmin, xmax = float(df_comp["MFLOPs"].min()), float(df_comp["MFLOPs"].max())
ax.set_xlim(xmin * 0.4, xmax * 2.5)
# ax.set_xticks(np.arange(0, 91, 10))

# 모델별 고정 색상 (사용자 지정)
MODEL_COLORS = {
    "CNN_LSTM":        "#1f3b73",   # navy
    "CNN_Transformer": "#808080",   # gray
    "ShuffleNetV2":    "#d62728",   # red (ShuffleNetV2 distinct)
}
model_names = df_comp["model"].tolist()
palette_models = {m: MODEL_COLORS.get(m, "#444444") for m in model_names}

size_min, size_max = 60, 420
sns.scatterplot(
    data=df_comp, x="MFLOPs", y="Accuracy",
    hue="model", size="Params_M", sizes=(size_min, size_max),
    palette=palette_models, legend=False, ax=ax
)

# --- FLOPs 저/고를 나누는 최적 임계선(수직선) ---
# KMeans(k=2)로 MFLOPs를 두 군집으로 나누고, 두 중심의 중간값을 임계로 사용
# --- 정확도와 FLOPs를 모두 고려한 '선형 분할선' (2D KMeans 기반 직선) ---
# 1) 표준화된 공간에서 KMeans(k=2)
try:
    from sklearn.cluster import KMeans
    from sklearn.preprocessing import StandardScaler

    XY = df_comp[["MFLOPs", "Accuracy"]].to_numpy()
    scaler = StandardScaler()
    Z = scaler.fit_transform(XY)  # z = ((x-μx)/σx, (y-μy)/σy)

    # 유효 포인트가 2개 이상일 때만
    if Z.shape[0] >= 2:
        km = KMeans(n_clusters=2, n_init=20, random_state=0).fit(Z)
        c1, c2 = km.cluster_centers_          # 두 군집 중심 (표준화 좌표)
        v = c2 - c1                           # 법선 벡터
        vx, vy = float(v[0]), float(v[1])
        const = 0.5 * (np.dot(c2, c2) - np.dot(c1, c1))  # 오른쪽 항

        # z-공간의 경계: v_x*zx + v_y*zy = const
        # 원좌표(y = m*x + b)로 변환
        mu_x, mu_y  = scaler.mean_[0],  scaler.mean_[1]
        sig_x, sig_y = np.sqrt(scaler.var_[0]), np.sqrt(scaler.var_[1])

        # vy가 거의 0이면 수평선; 아니면 직선 y = m x + b
        if abs(vy) < 1e-12:
            y_star = mu_y + sig_y * const / max(vy, 1e-12)
            ax.axhline(y_star, color="gray", lw=1.2, linestyle="--", alpha=0.95)
        else:
            m = - (sig_y * vx) / (sig_x * vy)
            b = (mu_y
                 + (sig_y * const) / vy
                 + (sig_y * vx * mu_x) / (sig_x * vy))
            x0, x1 = ax.get_xlim()
            y0, y1 = m * x0 + b, m * x1 + b
            ax.plot([x0, x1], [y0, y1], color="gray", lw=1.2, linestyle="--", alpha=0.95)

        # (옵션) 어느 쪽이 '저 FLOPs'인지 표시하고 싶다면:
        # 중심을 원좌표로 되돌려 비교
        c1_orig = np.array([mu_x + sig_x * c1[0], mu_y + sig_y * c1[1]])
        c2_orig = np.array([mu_x + sig_x * c2[0], mu_y + sig_y * c2[1]])
        low_center = c1_orig if c1_orig[0] < c2_orig[0] else c2_orig  # x(MFLOPs) 더 작은 쪽
        # ax.annotate("Lower FLOPs side", xy=(low_center[0], low_center[1]),
        #             xytext=(low_center[0], low_center[1] + 0.02*(ax.get_ylim()[1]-ax.get_ylim()[0])),
        #             arrowprops=dict(arrowstyle="->", color="crimson", lw=1.2),
        #             fontsize=9, color="crimson")
    else:
        # 포인트가 1개뿐이면 선형 분할선 의미 없음 → 수직 중앙값 대체
        t = float(np.median(df_comp["MFLOPs"]))
        ax.axvline(t, color="crimson", lw=2.2, alpha=0.95)

except Exception:
    # sklearn 미설치/에러 시: 수직 중앙값으로 폴백
    t = float(np.median(df_comp["MFLOPs"]))
    ax.axvline(t, color="crimson", lw=2.2, alpha=0.95)


# 커스텀 레전드(모델명 + Params(M))
vmin, vmax = df_comp["Params_M"].min(), df_comp["Params_M"].max()
def map_size(v):
    if vmax == vmin:
        return 0.5*(size_min + size_max)
    return size_min + (v - vmin) * (size_max - size_min) / (vmax - vmin)

df_leg = df_comp.sort_values("Params_M", ascending=False)
handles = []
for _, row in df_leg.iterrows():
    m, pM = row["model"], float(row["Params_M"])
    ms = np.sqrt(map_size(pM))  # Line2D는 지름(points)
    h = Line2D([], [], marker='o', linestyle='',
               markersize=ms,
               markerfacecolor=palette_models[m],
               markeredgecolor='white', markeredgewidth=0.6,
               label=f"{m} ({pM:.2f}M)")
    handles.append(h)

ax.legend(handles=handles, 
          title="Model (Params (M))",
          title_fontsize=14,   # 🔹 legend title 폰트 크기
          bbox_to_anchor=(0.995, 0.003), 
          loc="lower right",
          frameon=True, 
          handleheight=1.5, 
          labelspacing=0.4, 
          borderaxespad=0.5, 
          fontsize=14)  

# ---------- 그리드 ----------
# (0) 첫 번째: 카테고리(모델)마다 세로 그리드만
axes[0].grid(False)
axes[0].set_axisbelow(True)
axes[0].xaxis.grid(True, which="major", linestyle="--", linewidth=0.6, alpha=0.35)
axes[0].yaxis.grid(False)

# (1)(2) 두 번째/세 번째: x/y 둘 다
for a in (axes[1], axes[2]):
    a.grid(True, which="both", axis="both", linestyle="--", linewidth=0.6, alpha=0.35)
    a.set_axisbelow(True)

plt.subplots_adjust(wspace=0.15)
for a in axes:
    a.set_ylim(0.60, 1.00)
# 저장
plt.tight_layout()
out_path = f'./runs/{INPUT_KIND}/{RUN_TAG}/combined_jhys_unknown_scatter_{INPUT_KIND}.png'
plt.savefig(out_path, dpi=300)
plt.show()
print(f"[Saved] {out_path}")
